# Train the 3D U-Net baseline (M2-T03)

Trains experiment **EXP-0001** (`configs/training/EXP-0001.yaml`) on the BraTS 2021 **training** patients, checks it on
the **validation** patients, and logs everything in the team's MLflow store (`BrainMRI_Data/mlflow.db`, experiment
`BrainMRI_3D_UNet`). Locked test patients are never loaded.

**How to run:** **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
- **First session:** cleans the BraTS 2021 training and validation patients with Yasmin's pipeline (about 1 hour) and saves
  them to Drive as one archive, so later sessions only need about 10 minutes to restore them.
- **Training** stops cleanly after `MAX_HOURS`. **If Colab disconnects or the time runs out, just Run all again**: it
  resumes from the last checkpoint on Drive, in the same MLflow run.
- **Done when** the training cell prints `TRAINING FINISHED`. Checkpoints and final scores are in
  `BrainMRI_Data/runs/EXP-0001_unet3d_baseline/`.

In [ ]:
# 1. Settings, GPU and Drive
MAX_HOURS = 4.0              # training time per session; the final evaluation adds about 30 minutes at the end
BRANCH = "main"
SAVE_CLEANED_TO_DRIVE = True # keep an archive of the cleaned patients on Drive (about 18 GB) to skip cleaning next time

import os, sys, json, subprocess, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
CLEANED_TAR = BASE / "processed/brats2021_trainval_v1.tar"
OUT = BASE / "runs/EXP-0001_unet3d_baseline"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU: change the runtime type to T4 GPU before training"
!df -h /content | tail -1

In [ ]:
# 2. Code from the team repo, and the packages
if Path("/content/repo/.git").exists():
    !git -C /content/repo fetch -q origin && git -C /content/repo checkout -q {BRANCH} && git -C /content/repo pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/nhahub/NHA-5-187.git /content/repo
!git -C /content/repo log --oneline -1
!pip -q install "monai>=1.4" mlflow
sys.path.insert(0, "/content/repo/src")

In [ ]:
# 3. Cleaned BraTS 2021 training + validation patients in /content/processed/brats2021
#    Restored from the Drive archive when it exists; otherwise cleaned with Yasmin's pipeline (M1-T04) and archived.
processed = Path("/content/processed/brats2021")
marker = Path("/content/processed/.trainval_ready")
start = time.time()
if marker.exists():
    print("cleaned patients already on this Colab disk")
elif CLEANED_TAR.exists():
    print("restoring the cleaned patients from Drive ...")
    !mkdir -p /content/processed && tar -xf "{CLEANED_TAR}" -C /content/processed
    marker.touch()
else:
    print("cleaning the training + validation patients with pipeline/standardize.py (about 1 hour) ...")
    sys.path.insert(0, str(BASE / "pipeline"))
    import standardize as S
    jobs = S.regenerate(splits=("train", "val"), datasets=("brats2021",))   # test patients stay locked
    assert (jobs.status != "failed").all(), jobs[jobs.status == "failed"].head()
    marker.touch()
    if SAVE_CLEANED_TO_DRIVE:
        CLEANED_TAR.parent.mkdir(parents=True, exist_ok=True)
        !tar -cf "{CLEANED_TAR}" -C /content/processed brats2021
print(f"{len(list(processed.glob('BraTS2021_*')))} cleaned patients ready ({(time.time() - start) / 60:.0f} min)")

In [ ]:
# 4. Train (resumes automatically from OUT/last.pt) and, when all epochs are done, evaluate on all validation patients
%cd /content/repo
!python scripts/train.py --exp configs/training/EXP-0001.yaml --max-hours {MAX_HOURS}

In [ ]:
# 5. Results from MLflow
import shutil, mlflow, pandas as pd, matplotlib.pyplot as plt
from mlflow.tracking import MlflowClient
shutil.copyfile(BASE / "mlflow.db", "/content/mlflow_read.db")   # SQLite directly on Drive fails ("disk I/O error")
uri = "sqlite:////content/mlflow_read.db"
client = MlflowClient(tracking_uri=uri)
exp = client.get_experiment_by_name("BrainMRI_3D_UNet")
run = client.search_runs([exp.experiment_id], filter_string="tags.mlflow.runName = 'EXP-0001'",
                         order_by=["attributes.start_time DESC"], max_results=1)[0]
print("run", run.info.run_id, "| status", run.info.status, "|", run.data.tags.get("state"))
final = {k: round(v, 4) for k, v in sorted(run.data.metrics.items()) if k.startswith("final_")}
print(json.dumps(final, indent=1) if final else "final evaluation not run yet (training not finished)")

hist = lambda key: pd.DataFrame([(m.step, m.value) for m in client.get_metric_history(run.info.run_id, key)], columns=["epoch", key])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
loss = hist("train_loss"); ax[0].plot(loss.epoch, loss.train_loss); ax[0].set(title="training loss", xlabel="epoch")
for r in ("TC", "WT", "ET"):
    d = hist(f"val_dice_{r}")
    if len(d): ax[1].plot(d.epoch, d[f"val_dice_{r}"], marker="o", label=r)
ax[1].set(title="validation Dice (40 patients)", xlabel="epoch", ylim=(0, 1)); ax[1].legend(); plt.show()

In [ ]:
# 6. Finish writing to Drive
drive.flush_and_unmount()